# 从ratings_Sports_and_Outdoors.csv文件中提取U-I交互图, 5-core后重新编号
- Extracting U-I interactions and performing 5-core, re-indexing
- dataset located at: http://jmcauley.ucsd.edu/data/amazon/links.html, rating only file in "Small" subsets for experimentation

In [1]:
import os, csv
import pandas as pd

In [2]:
os.chdir('/home/minhle/CaMRec/raw_data')
os.getcwd()

'/home/minhle/CaMRec/raw_data'

## 先5-core过滤
## 5-core filtering

In [3]:
df = pd.read_csv('ratings_Baby.csv', names=['userID', 'itemID', 'rating', 'timestamp'], header=None)
print(f'shape: {df.shape}')
df[:5]

shape: (915446, 4)


,userID,itemID,rating,timestamp
0,A28O3NP6WR5517,0188399313,5.0,1369612800
1,AX0M1Z6ZWO52J,0188399399,5.0,1365465600
2,A1KD7N84L7NIUT,0188399518,4.0,1392336000
3,A29CUDEIF4X1UO,0188399518,3.0,1373241600
4,A32592TYN6C9EM,0316967297,4.0,1378425600


In [4]:
k_core = 5
learner_id, course_id, tmstmp_str = 'userID', 'itemID', 'timestamp'

df.dropna(subset=[learner_id, course_id, tmstmp_str], inplace=True)
df.drop_duplicates(subset=[learner_id, course_id, tmstmp_str], inplace=True)
print(f'After dropped: {df.shape}')
df[:3]

After dropped: (915446, 4)


,userID,itemID,rating,timestamp
0,A28O3NP6WR5517,0188399313,5.0,1369612800
1,AX0M1Z6ZWO52J,0188399399,5.0,1365465600
2,A1KD7N84L7NIUT,0188399518,4.0,1392336000


In [5]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 5, 5

def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num}
    print(f'{len(ids)} illegal_ids_by_inter_num, field={field}')

    return ids


def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(df, field=learner_id, max_num=None, min_num=min_u_num)
        ban_items = get_illegal_ids_by_inter_num(df, field=course_id, max_num=None, min_num=min_i_num)
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if learner_id:
            dropped_inter |= df[learner_id].isin(ban_users)
        if course_id:
            dropped_inter |= df[course_id].isin(ban_items)
        print(f'{len(dropped_inter)} dropped interactions')
        df.drop(df.index[dropped_inter], inplace=True)



## k-core

In [6]:
filter_by_k_core(df)
print(f'k-core shape: {df.shape}')
print(f'shape after k-core: {df.shape}')
df[:2]

504235 illegal_ids_by_inter_num, field=userID
42628 illegal_ids_by_inter_num, field=itemID
915446 dropped interactions
3078 illegal_ids_by_inter_num, field=userID
10456 illegal_ids_by_inter_num, field=itemID
216360 dropped interactions
4107 illegal_ids_by_inter_num, field=userID
446 illegal_ids_by_inter_num, field=itemID
184242 dropped interactions
349 illegal_ids_by_inter_num, field=userID
644 illegal_ids_by_inter_num, field=itemID
167664 dropped interactions
504 illegal_ids_by_inter_num, field=userID
53 illegal_ids_by_inter_num, field=itemID
163929 dropped interactions
53 illegal_ids_by_inter_num, field=userID
79 illegal_ids_by_inter_num, field=itemID
161727 dropped interactions
66 illegal_ids_by_inter_num, field=userID
8 illegal_ids_by_inter_num, field=itemID
161209 dropped interactions
4 illegal_ids_by_inter_num, field=userID
11 illegal_ids_by_inter_num, field=itemID
160915 dropped interactions
13 illegal_ids_by_inter_num, field=userID
2 illegal_ids_by_inter_num, field=itemID
16085

,userID,itemID,rating,timestamp
19,A1HK2FQW6KXQB2,097293751X,5.0,1373932800
22,A19K65VY14D13R,097293751X,5.0,1372464000


## Re-index

In [7]:
df.reset_index(drop=True, inplace=True)

In [8]:

i_mapping_file = 'i_id_mapping.csv'
u_mapping_file = 'u_id_mapping.csv'

splitting = [0.8, 0.1, 0.1]
uid_field, iid_field = learner_id, course_id

uni_users = pd.unique(df[uid_field])
uni_items = pd.unique(df[iid_field])

# start from 0
u_id_map = {k: i for i, k in enumerate(uni_users)}
i_id_map = {k: i for i, k in enumerate(uni_items)}

df[uid_field] = df[uid_field].map(u_id_map)
df[iid_field] = df[iid_field].map(i_id_map)
df[uid_field] = df[uid_field].astype(int)
df[iid_field] = df[iid_field].astype(int)

# dump
rslt_dir = '../data'
u_df = pd.DataFrame(list(u_id_map.items()), columns=['user_id', 'userID'])
i_df = pd.DataFrame(list(i_id_map.items()), columns=['asin', 'itemID'])

u_df.to_csv(os.path.join(rslt_dir, u_mapping_file), sep='\t', index=False)
i_df.to_csv(os.path.join(rslt_dir, i_mapping_file), sep='\t', index=False)
print(f'mapping dumped...')

mapping dumped...


In [9]:

# =========2. splitting
print(f'splitting ...')
tot_ratio = sum(splitting)
# remove 0.0 in ratios
ratios = [i for i in splitting if i > .0]
ratios = [_ / tot_ratio for _ in ratios]
split_ratios = np.cumsum(ratios)[:-1]

#df[tmstmp_str] = df[tmstmp_str].map(lambda x: datetime.strptime(x, "%Y-%m-%dT%H:%M:%SZ"))
split_ratios

splitting ...


array([0.8, 0.9])

In [12]:
ts_id = 'timestamp'

split_timestamps = list(np.quantile(df[ts_id], split_ratios))
# get df training dataset unique users/items
df_train = df.loc[df[ts_id] < split_timestamps[0]].copy()
df_val = df.loc[(split_timestamps[0] <= df[ts_id]) & (df[ts_id] < split_timestamps[1])].copy()
df_test = df.loc[(split_timestamps[1] <= df[ts_id])].copy()

x_label, rslt_file = 'x_label', 'baby-indexed.inter'
df_train[x_label] = 0
df_val[x_label] = 1
df_test[x_label] = 2
temp_df = pd.concat([df_train, df_val, df_test])
temp_df = temp_df[[learner_id, course_id, 'rating', ts_id, x_label]]
print(f'columns: {temp_df.columns}')

temp_df.columns = [learner_id, course_id, 'rating', ts_id, x_label]

temp_df.to_csv(os.path.join(rslt_dir, rslt_file), sep='\t', index=False)
temp_df[:5]
#print('done!')

columns: Index(['userID', 'itemID', 'rating', 'timestamp', 'x_label'], dtype='object')


,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1373932800,0
1,1,0,5.0,1372464000,0
3,3,0,5.0,1376697600,0
6,6,0,5.0,1374019200,0
7,7,0,5.0,1359244800,0


## Reload

In [14]:
indexed_df = pd.read_csv(os.path.join(rslt_dir, rslt_file), sep='\t')
print(f'shape: {indexed_df.shape}')
indexed_df[:4]

shape: (160792, 5)


,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1373932800,0
1,1,0,5.0,1372464000,0
2,3,0,5.0,1376697600,0
3,6,0,5.0,1374019200,0


In [15]:
u_uni = indexed_df[learner_id].unique()
c_uni = indexed_df[course_id].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 19445
# of unique courses: 7050
min/max of unique learners: 0/19444
min/max of unique courses: 0/7049
